# GRADED CHALLENGE 6
***Link Google Slide is in the last cell of this notebook***

## I. INTRODUCTION
**NAME      :** Khoizuron Syakira\
**BATCH     :** CODA RMT-023

**OBJECTIVE :**
Build one sales data warehouse so all three main questions can be answered form the same tables using queries.

**DATA SCOPE :**
All data history from January 2019 to November 1, 2026. Because 2026 is not a full yet.

## II. CORE BUSINESS AND DATA MODELLING
The look sells fashion products through a website. Every one item bought in an order is saved as a sales record with its price, cost, profit and status.

**Sales Flow**
User regitration - choose products - orders - goods handled - shipped - delivered or returned.

**The Problem:**
The raw data is spread across 7 tables which causes hard to analyze cause needed many joins, which take times and causing different results.
There are two main questions:
- Which category makes a real profit?
- What is the actual revenue for every year?

for answering the problem we only need to extract only 4 essential tables not including inventory_items, events and distribution_center

**Data Modelling**
We are using star schema with with 1 fact table and 4 dimention table. Fact table will consist `fact_sales` while the dimemtion tables consist `dim_user, dim_products, dim_order, dim_date`



### SETUP
We need to setup the environment first and intialize pySpark to run in this notebook. We set `JAVA_HOME` path from conda environment, load database credential in the .env file using `dotenv` and then start `sparkSession` with postgres JDBC driver and set the timezone to UTC.

In [1]:
import os, sys

# Java from env conda (so it can read while running through vs code)
os.environ.setdefault("JAVA_HOME", os.path.join(sys.prefix, "Library" if os.name == "nt" else "lib/jvm"))

'/opt/anaconda3/envs/dbt313/lib/jvm'

In [2]:
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

load_dotenv('sql/.env', override=True)
print(os.getenv('PG_DB'))

spark = SparkSession.builder \
    .appName("ETLSuperstore") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.14") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

thelook_dwh


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/05 22:27:10 WARN Utils: Your hostname, Kyras-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.18.101 instead (on interface en0)
26/11/05 22:27:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/anaconda3/envs/dbt313/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/kyras/.ivy2.5.2/cache
The jars for the packages stored in: /Users/kyras/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-04693138-8f6a-43fd-8aec-eaf785bf985d;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.14 in central
	found org.checkerframework#checker-qual;3.55.1 in central
:: resolution report :: resolve 94ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

## III. EXTRACT

**READ THE CSV** with two helper function
- `read_csv` reads one CSV file into a pySpark dataframe
- `to_snake_case` makes all the column names lowercase with an underscore so it is consistent.

In [3]:
def read_csv(file_name):
    return spark.read.csv(f'data/{file_name}', header=True, inferSchema=True, escape='"')

def to_snake_case(df):
    new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in df.columns]
    return df.toDF(*new_names)

order_items         = to_snake_case(read_csv('order_items.csv'))
orders              = to_snake_case(read_csv('orders.csv'))
users               = to_snake_case(read_csv('users.csv'))
products            = to_snake_case(read_csv('products.csv'))

order_items.count(), orders.count(), users.count(), products.count()

(180746, 124452, 100000, 29120)

Results from extract showing that 5 files are loaded with the expected row counts and rady to be transform

## IV. TRANSFORM
This step changes the raw extracted data into tables that ready to load into the data warehouse.


### CHECK DIRTY DATA
The data that have problems like wrong data types, missing values, different spellings, duplicates.

In [4]:
order_items.groupby('status').count().show()
orders.groupby('status').count().show()
users.groupby('gender').count().show()
users.groupby('traffic_source').count().show()
order_items.select(F.min('sale_price'), F.max('sale_price')).show()
products.select(F.min('cost'), F.max('cost')).show()

+----------+-----+
|    status|count|
+----------+-----+
|  Returned|18039|
|   Shipped|54276|
|Processing|35822|
|  Complete|45532|
| Cancelled|27077|
+----------+-----+

+----------+-----+
|    status|count|
+----------+-----+
|  Returned|12396|
|   Shipped|37308|
|Processing|24755|
|  Complete|31359|
| Cancelled|18634|
+----------+-----+

+------+-----+
|gender|count|
+------+-----+
|     F|49805|
|     M|50195|
+------+-----+

+--------------+-----+
|traffic_source|count|
+--------------+-----+
|       Organic|14692|
|       Display| 3959|
|         Email| 5021|
|        Search|70294|
|      Facebook| 6034|
+--------------+-----+

+------------------+---------------+
|   min(sale_price)|max(sale_price)|
+------------------+---------------+
|0.0199999995529651|          999.0|
+------------------+---------------+

+------------------+-----------------+
|         min(cost)|        max(cost)|
+------------------+-----------------+
|0.0082999997779726|557.1510021798313|
+--------------

Based on the outpus, there are some cleaning steps that needed to be done. Renaming the primary key columns, cleaning the text like removing extra spaces and standard casing, rounding floating number to only 2 decimal places, last removing duplicate rows and fill missing string values with 'Unknown' and filtering out invalid foreign keys

### CLEANING
The data willl be clean after we check and we rename it to match the tables in postgres and to check the UTC is the same.

In [5]:
print(spark.conf.get('spark.sql.session.timeZone'))
order_items.filter(F.col('id') == 3671).select('id', 'created_at').show(truncate=False)

UTC
+----+-------------------+
|id  |created_at         |
+----+-------------------+
|3671|2026-04-11 23:26:05|
+----+-------------------+



In [6]:
users_clean = (users
    .drop_duplicates()
    .withColumnRenamed('id', 'user_id')
    .withColumnRenamed('created_at', 'user_created_at')
    .withColumn('first_name', F.trim('first_name'))
    .withColumn('last_name', F.trim('last_name'))
    .withColumn('gender', F.upper(F.trim('gender')))
    .withColumn('city', F.trim('city'))
    .withColumn('state', F.trim('state'))
    .withColumn('country', F.trim('country'))
    .withColumn('traffic_source', F.initcap(F.trim('traffic_source')))
    .fillna({'first_name': 'Unknown', 'last_name': 'Unknown', 'gender': 'Unknown', 'city': 'Unknown', 'state': 'Unknown', 'country': 'Unknown', 'traffic_source': 'Unknown'})
    .dropna(subset=['user_id'])
    .drop_duplicates(subset=['user_id']))

users_clean.groupby('gender').count().show()

+------+-----+
|gender|count|
+------+-----+
|     F|49805|
|     M|50195|
+------+-----+



In [7]:
orders_clean = (orders
    .drop_duplicates()
    .withColumnRenamed('status', 'order_status')
    .withColumn('order_status', F.initcap(F.trim('order_status')))
    .withColumn('num_of_item', F.col('num_of_item').cast('int'))
    .fillna({'order_status': 'Unknown'})
    .dropna(subset=['order_id'])
    .drop_duplicates(subset=['order_id']))

orders_clean.groupby('order_status').count().show()

+------------+-----+
|order_status|count|
+------------+-----+
|    Returned|12396|
|     Shipped|37308|
|  Processing|24755|
|    Complete|31359|
|   Cancelled|18634|
+------------+-----+



In [8]:
products_clean = (products
    .drop_duplicates()
    .withColumnRenamed('id', 'product_id')
    .withColumnRenamed('name', 'product_name')
    .withColumnRenamed('cost', 'product_cost')
    .withColumn('product_name', F.trim('product_name'))
    .withColumn('brand', F.trim('brand'))
    .withColumn('category', F.trim('category'))
    .withColumn('department', F.initcap(F.trim('department')))
    .withColumn('product_cost', F.round(F.col('product_cost').cast('double'), 2))
    .withColumn('retail_price', F.round(F.col('retail_price').cast('double'), 2))
    .fillna({'product_name': 'Unknown', 'brand': 'Unknown', 'category': 'Unknown', 'department': 'Unknown'})
    .dropna(subset=['product_id'])
    .drop_duplicates(subset=['product_id']))

products_clean.groupby('category').count().show(30, truncate=False)
products.count(), products_clean.count()

+-----------------------------+-----+
|category                     |count|
+-----------------------------+-----+
|Dresses                      |955  |
|Swim                         |1798 |
|Outerwear & Coats            |1420 |
|Tops & Tees                  |1868 |
|Suits & Sport Coats          |739  |
|Socks & Hosiery              |666  |
|Blazers & Jackets            |561  |
|Suits                        |188  |
|Active                       |1432 |
|Pants                        |1041 |
|Pants & Capris               |613  |
|Clothing Sets                |37   |
|Leggings                     |564  |
|Skirts                       |367  |
|Intimates                    |2363 |
|Fashion Hoodies & Sweatshirts|1866 |
|Plus                         |758  |
|Accessories                  |1559 |
|Maternity                    |898  |
|Underwear                    |1088 |
|Sleep & Lounge               |1771 |
|Shorts                       |1765 |
|Sweaters                     |1737 |
|Jumpsuits &

(29120, 29120)

In [9]:
order_items_clean = (order_items
    .drop_duplicates()
    .withColumnRenamed('id', 'order_item_id')
    .withColumnRenamed('status', 'item_status')
    .withColumnRenamed('created_at', 'order_created_at')
    .withColumn('item_status', F.initcap(F.trim('item_status')))
    .withColumn('sale_price', F.round(F.col('sale_price'), 2))
    .dropna(subset=['order_item_id', 'order_id', 'user_id', 'product_id', 'order_created_at', 'sale_price'])
    .drop_duplicates(subset=['order_item_id']))

order_items_clean.groupby('item_status').count().show()

+-----------+-----+
|item_status|count|
+-----------+-----+
|   Returned|18039|
|    Shipped|54276|
| Processing|35822|
|   Complete|45532|
|  Cancelled|27077|
+-----------+-----+



We have cleaned the four tables by removing duplicate and null records, stadirizing column names, filling missing values and rounding monetary figures. Next cleaning order that has no Product, User, or Order at all. The one that key is in the other table. Items like that will get decline by foreign key in postgres, so it will be mark as `left-anti`

In [10]:
order_items_clean.join(products_clean, on='product_id', how='left_anti').count()
order_items_clean.join(users_clean, on='user_id', how='left_anti').count()
order_items_clean.join(orders_clean, on='order_id', how='left_anti').count()

0

In [11]:
order_items_valid = (order_items_clean
    .join(products_clean[['product_id']], on='product_id', how='inner')
    .join(users_clean[['user_id']], on='user_id', how='inner')
    .join(orders_clean[['order_id']], on='order_id', how='inner'))

We checked all 180,746 rows using left-anti join and found no missing product, user, or order IDs, so order_items_valid is completely safe to use for building our fact and dimention tables.

## DIMENTION TABLE
Every dimention is a clean table with needed column only so the list would be the same as the one in postgres

In [12]:
dim_users = users_clean[['user_id', 'first_name', 'last_name', 'age', 'gender', 'city', 'state', 'country', 'traffic_source', 'user_created_at']]
dim_products = products_clean[['product_id', 'product_name', 'brand', 'category', 'department', 'sku', 'retail_price']]
dim_orders = orders_clean[['order_id', 'order_status', 'num_of_item']]

dim_users.show(3)
dim_products.show(3)
dim_orders.show(3)

+-------+----------+---------+---+------+--------+--------------+-------------+--------------+-------------------+
|user_id|first_name|last_name|age|gender|    city|         state|      country|traffic_source|    user_created_at|
+-------+----------+---------+---+------+--------+--------------+-------------+--------------+-------------------+
|     26|  Jennifer|   Lucero| 19|     F|Hengshui|       Beijing|        China|        Search|2020-10-02 11:16:00|
|     27|     Wayne|   Walker| 34|     M|  Conway|South Carolina|United States|         Email|2023-03-12 18:07:00|
|     28| Christine| Townsend| 40|     F|  Aurora|      Colorado|United States|        Search|2024-03-05 12:02:00|
+-------+----------+---------+---+------+--------+--------------+-------------+--------------+-------------------+
only showing top 3 rows
+----------+--------------------+------------------+-----------+----------+--------------------+------------+
|product_id|        product_name|             brand|   catego

We succeded creating sim_users, dim_products and dim_orders using only the essential column to match out postgres database columns

In [13]:
dates = order_items_valid.select(F.to_date('order_created_at').alias('full_date')) \
    .union(order_items_valid.select(F.to_date('shipped_at').alias('full_date'))) \
    .dropna() \
    .distinct()

dim_date = dates \
    .withColumn('date_key', F.date_format('full_date', 'yyyyMMdd').cast('int')) \
    .withColumn('year', F.year('full_date')) \
    .withColumn('quarter', F.quarter('full_date')) \
    .withColumn('month', F.month('full_date')) \
    .withColumn('month_name', F.date_format('full_date', 'MMMM')) \
    .withColumn('day_name', F.date_format('full_date', 'EEEE'))
dim_date.show(5)

+----------+--------+----+-------+-----+----------+--------+
| full_date|date_key|year|quarter|month|month_name|day_name|
+----------+--------+----+-------+-----+----------+--------+
|2026-10-09|20261009|2026|      4|   10|   October|  Friday|
|2026-08-18|20260818|2026|      3|    8|    August| Tuesday|
|2026-02-13|20260213|2026|      1|    2|  February|  Friday|
|2023-06-22|20230622|2023|      2|    6|      June|Thursday|
|2022-07-31|20220731|2022|      3|    7|      July|  Sunday|
+----------+--------+----+-------+-----+----------+--------+
only showing top 5 rows


We created dim_date by extracting unique dates from order and shipping timestamps and breaking them up into year, quarter, month and day details.

## FACT TABLE
We created fact_sales from joining order_items_valid with products_clean, generating date_key, and calculating profit from (sale_proce-product_cost)

In [14]:
fact_sales = (order_items_valid
    .join(products_clean.select('product_id', 'product_cost'), on='product_id', how='left')
    .withColumn('date_key', F.date_format('order_created_at', 'yyyyMMdd').cast('int')) \
    .withColumn('profit', F.round(F.col('sale_price')-F.col('product_cost'), 2)) \
    .select('order_item_id', 'order_id', 'user_id', 'product_id', 'date_key', 'item_status',
            'order_created_at', 'shipped_at', 'delivered_at', 'returned_at', 'sale_price', 
            'product_cost', 'profit'))
fact_sales.show(5)

+-------------+--------+-------+----------+--------+-----------+--------------------+--------------------+--------------------+--------------------+----------+------------+------+
|order_item_id|order_id|user_id|product_id|date_key|item_status|    order_created_at|          shipped_at|        delivered_at|         returned_at|sale_price|product_cost|profit|
+-------------+--------+-------+----------+--------+-----------+--------------------+--------------------+--------------------+--------------------+----------+------------+------+
|           26|      17|     12|     19306|20261028|    Shipped|2026-10-28 13:42:...|2026-10-30 09:32:...|                NULL|                NULL|      56.0|       25.93| 30.07|
|           27|      18|     12|     24861|20261028|   Returned|2026-10-28 22:15:...|2026-10-31 10:05:...|2026-11-03 13:28:...|2026-11-04 04:25:...|     18.95|       11.12|  7.83|
|           28|      19|     13|     17413|20261028|  Cancelled| 2026-10-28 14:09:04|               

In [15]:
# PostgreSQL JDBC Connection (isi ada di file .env)
postgres_url = f"jdbc:postgresql://{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
postgres_properties = {
    "user": os.getenv("PG_USER"),
    "password": os.getenv("PG_PASSWORD"),
    "driver": "org.postgresql.Driver"
}

In [16]:
# dimension dulu, baru fact
dim_users.write.jdbc(url=postgres_url, table="dim_users", mode="append", properties=postgres_properties)
dim_products.write.jdbc(url=postgres_url, table="dim_products", mode="append", properties=postgres_properties)
dim_orders.write.jdbc(url=postgres_url, table="dim_orders", mode="append", properties=postgres_properties)
dim_date.write.jdbc(url=postgres_url, table="dim_date", mode="append", properties=postgres_properties)
fact_sales.write.jdbc(url=postgres_url, table="fact_sales", mode="append", properties=postgres_properties)

26/11/05 22:28:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## V. LOAD
The data is cleaned and ready to load into the data warehouse.

This is to verify our question about which category makes the real profit?

In [17]:
sql_query = """SELECT
    p.category,
    ROUND(SUM(f.profit), 2)::float AS total_profit
FROM fact_sales f
JOIN dim_products p ON f.product_id = p.product_id
WHERE f.item_status NOT IN('Cancelled', 'Returned')
GROUP BY 1
ORDER BY 2 DESC"""
query = f"({sql_query}) AS subquery"


df = spark.read.jdbc(url=postgres_url, table=query, properties=postgres_properties)
df.show()

+--------------------+------------+
|            category|total_profit|
+--------------------+------------+
|   Outerwear & Coats|   551923.98|
|               Jeans|   440902.17|
|            Sweaters|   323357.81|
| Suits & Sport Coats|   293387.81|
|                Swim|   235549.11|
|Fashion Hoodies &...|   233516.38|
|      Sleep & Lounge|   212458.72|
|              Active|   195892.25|
|              Shorts|    194202.7|
|         Accessories|   194177.21|
|             Dresses|    188942.9|
|               Pants|   174924.42|
|         Tops & Tees|   161580.06|
|           Intimates|   160411.92|
|   Blazers & Jackets|   133627.59|
|           Maternity|   105357.56|
|           Underwear|    79592.01|
|      Pants & Capris|    68188.71|
|                Plus|    61243.28|
|              Skirts|    46200.17|
+--------------------+------------+
only showing top 20 rows


The result is shwoing that outerwar and coats brings the highest profit to us and followed by jeans. 

This is to verify our question about how much is our actual revenue every year?

In [18]:
sql_query2 = """SELECT
    d.year,
    ROUND(SUM(f.sale_price), 2)::float AS revenue
FROM fact_sales f
JOIN dim_date d ON f.date_key = d.date_key
WHERE f.item_status NOT IN('Cancelled', 'Returned')
GROUP BY 1
ORDER BY 1"""
query2 = f"({sql_query2}) AS subquery"


df = spark.read.jdbc(url=postgres_url, table=query2, properties=postgres_properties)
df.show()

+----+----------+
|year|   revenue|
+----+----------+
|2019|  65600.07|
|2020| 214668.04|
|2021| 386000.99|
|2022| 597218.03|
|2023| 843354.67|
|2024|1249604.79|
|2025|1828599.86|
|2026|2913871.63|
+----+----------+



The result shows that our revenue is growing every year from $65,600.07 to $2,913,871.63

## VI. FINISH
We stop the sparkSession so it stop running in the background and use our memory

In [19]:
spark.stop()

***Link Google Slide:https://docs.google.com/presentation/d/10mBjrMPftR-e3SM4XRea-Lnq-bCHe0ghfmPE1qfFgRo/edit?usp=sharing***